In [1]:
bundle_path: "EOInput" = "test_packaging3"
selection = "ensemble_simple"
input_dataset_path: "EOInput" = "input/elbe_bunthaus"
mode = "from_date"
start_date: str
# output_path: "EOInput" = "outputs/predictions.csv"

xcengine_config = dict(
  # ...
    include_directory=True,
    build_includes=["required_dist/"]  # paths relative to notebook
)

In [2]:
from pathlib import Path
from typing import get_type_hints

import pandas as pd
import pystac
from datetime import datetime, timezone

from pftnc.inference_config import InferenceConfig
from pftnc.predict.inference import run_inference

In [3]:
output_path = "predictions.csv"

In [4]:
mode_type = get_type_hints(InferenceConfig)["mode"]
model_type = get_type_hints(InferenceConfig)["model"]
selection_type = get_type_hints(model_type)["selection"]

In [5]:
selection: selection_type = selection
mode: mode_type = mode

In [6]:
def extract_assets_from_catalog(catalog: pystac.Catalog, asset_key: str) -> list[pystac.Asset]:
    """
    Returns all assets with a given key from the items of a catalog.
    """
    assets = []
    for item in catalog.get_all_items():
        if (asset := item.assets.get(asset_key)) is not None:
            assets.append(asset)

    return assets

def get_catalog(inp: Path | str) -> pystac.Catalog:
    p = Path(inp) / "catalog.json"
    catalog = pystac.Catalog.from_file(p)
    catalog.make_all_asset_hrefs_absolute()
    return catalog

In [7]:
catalog_elbe_bunthaus = get_catalog(input_dataset_path)

In [8]:
asset_id  = "elbe_bunthaus"
fpath_elbe_bunthaus = next(iter(extract_assets_from_catalog(catalog_elbe_bunthaus, asset_id))).href

In [9]:
config = InferenceConfig(
      model={
          "bundle_path": Path(bundle_path),
          "selection": selection,
      },
      input_dataset={
          "path": Path(fpath_elbe_bunthaus),
      },
      mode=mode,
      start_date="2023-12-01",
      output_path=Path(output_path),
  )

In [10]:
predictions = run_inference(config)
print("pftnc predictions successfully generated")

Generating rolling features:   0%|          | 0/360 [00:00<?, ?it/s]

pftnc predictions successfully generated


In [11]:
base_path = Path("./datasets_saved")
base_path.mkdir(exist_ok=True, parents=True)

In [12]:
def generate_stac(df: pd.DataFrame):
    from shapely.geometry import Point
    
    lat = df["site_lat"].iloc[0]
    lon = df["site_lon"].iloc[0]
    
    point = Point(lon, lat)
    
    layout_strategy = pystac.layout.CustomLayoutStrategy(
            item_func=lambda item, parent: Path(parent) / base_path.name / f"{item.id}.json"
        )
    
    item = pystac.Item(
        "pftnc_predictions",
        geometry = point.__geo_interface__,
        bbox = [lon, lat, lon, lat],
        datetime=datetime.now(tz=timezone.utc),
        properties={},
    )
    
    catalog = pystac.Catalog(
        "catalog", "Pythoplankton Functional Types Nowcasting Predictions", 
        strategy=layout_strategy,
        catalog_type = pystac.CatalogType.SELF_CONTAINED
    )
    catalog.add_item(item)
    catalog.normalize_and_save("catalog.json")
    

In [13]:
generate_stac(predictions)